### Hallucination Detection using NLIm

In [ ]:
import numpy as np

### Loading the Data

In [ ]:
from pathlib import Path

In [ ]:
current_working_directory = Path.cwd().parent.parent

In [ ]:
current_working_directory

In [ ]:
data_directory = current_working_directory.joinpath("datasets/halifax_intermediaries/")
data_directory.exists()

In [ ]:
first_iteration_path = data_directory.joinpath("first_iteration_answers.csv")

In [ ]:
second_iteration_path = data_directory.joinpath("second_iteration.csv")

In [ ]:
import pandas as pd

In [ ]:
first_iteration_df = pd.read_csv(first_iteration_path, sep="|", index_col=0)

In [ ]:
second_iteration_question_df = pd.read_csv(second_iteration_path, sep="|", engine="python")

In [ ]:
first_iteration_df.columns

In [ ]:
from ast import literal_eval

In [ ]:
def filter_citations(df):
    """ Retrieve the chunks that are only present in the citations list """
    df['citations'] = df['citations'].apply(literal_eval)
    df = df.explode('citations')
    df = df[df.citations == df["chunk_index"]]
    return df

In [ ]:
second_iteration_question_df = second_iteration_question_df.set_index(['Question', 'generated_answer', 'Answer', 'Conversation_Index',
                                        'is_correct'])

In [ ]:
second_iteration_question_df["chunk_index"] = second_iteration_question_df.groupby(level=[0,1,2,3]).cumcount() + 1

In [ ]:
second_iteration_question_df = filter_citations(second_iteration_question_df)

In [ ]:
second_iteration_question_df.head()

In [ ]:
second_iteration_question_df = second_iteration_question_df.reset_index()

In [ ]:
questions_df = pd.concat([first_iteration_df, second_iteration_question_df[['Question', 'generated_answer', 'Answer', 'distance', 'content']],
                          ], axis="rows")

In [ ]:
qa_pairs_df = questions_df

### Split Each Answer into sentences.

In [ ]:
from nltk.tokenize import sent_tokenize

In [ ]:
import re

In [ ]:
reference_regex = r'[\s\(\[]*(?:Documents?\s*)?\d+(?:\s*,\s*\d+)*[\s\)\]\.,]*' # Regex to remove any occurence of document references



In [ ]:
qa_pairs_df.loc[:, "generated_answer_sentence"] = qa_pairs_df["generated_answer"].apply(sent_tokenize)


In [ ]:
qa_pairs_df.head()

In [ ]:
qa_pairs_df = qa_pairs_df.explode(
    "generated_answer_sentence").reset_index(drop=True)
qa_pairs_df = qa_pairs_df.dropna(subset=["generated_answer_sentence"])

In [ ]:
qa_pairs_df.head()

### Entailment Prediction

In [ ]:
qa_pairs_df.head()

With the first approach of using a veto approach we have declined 46 question in total.

In [ ]:
import numpy as np
import random

In [ ]:
query_df = qa_pairs_df[qa_pairs_df.Question.str.contains(
    'Hi Anjali, My client is purchasing a new resid')]

In [ ]:
query_df.head()

In [ ]:
def style_max_element(cell, labels_to_color: dict):
    """
    Determines the style and display value based on the position of the max element.
    If the max element is at index 0, color is red; index 1, color is green; index 2, color is blue.
    Returns:
    - style (str): 'background-color: color' CSS string.
    - display_value (float): The maximum value in the list.
    """
    if not isinstance(cell, list) or len(cell) != 3:
        return '', cell

    max_val = max(cell)
    max_index = cell.index(max_val)

    color = labels_to_color.get(max_index)

    return f'background-color: {color}', f'{max_val:.2f}'

def color_cell(cell, labels_to_color={0: 'red', 1: 'green', 2: 'blue'}):
    """Applies the background color style."""
    style, _ = style_max_element(cell, labels_to_color=labels_to_color)
    return style


def display_value(cell, labels_to_color={0: 'red', 1: 'green', 2: 'blue'}):
    """Applies the display value format (max element rounded to 2 decimals)."""
    _, value = style_max_element(cell, labels_to_color=labels_to_color)
    return value

#### Trying Albert Xlarge

In [ ]:
model_name = "tals/albert-xlarge-vitaminc-mnli"

In [ ]:
model_cache = Path().cwd().joinpath("models")

In [ ]:
from sentence_transformers import CrossEncoder

xlarge_model = CrossEncoder(model_name, cache_folder=model_cache)

### Sentence Level Split

In [ ]:
sentence_level_scores = xlarge_model.predict(
    qa_pairs_df[["content", "generated_answer_sentence"]].values.tolist(), # the content, support the sentence, not the opposite
    apply_softmax=True,
    batch_size=8)

In [ ]:
sentence_level_scores.shape


In [ ]:
alberta_labels = xlarge_model.config.id2label

In [ ]:
alberta_labels

In [ ]:
qa_pairs_df["entailment_score_alberta"] = (
    sentence_level_scores.round(2) * 100).tolist()

In [ ]:
entailment_score_alberta = qa_pairs_df.entailment_score_alberta.apply(
    pd.Series)
entailment_score_alberta.columns = alberta_labels.values()

In [ ]:
alberta_labels.values()

In [ ]:
entailment_score_alberta.head()

In [ ]:
qa_pairs_df = pd.concat([qa_pairs_df, entailment_score_alberta
                         ], axis=1)

In [ ]:
qa_pairs_df.head()

In [ ]:
qa_pairs_df_xberta = qa_pairs_df[["Question", "generated_answer_sentence", "content", "entailment_score_alberta", "SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]]

In [ ]:
qa_pairs_df_xberta = qa_pairs_df_xberta.rename({
    "SUPPORTS": "entailment",
    "REFUTES": "contradiction",
    "NOT ENOUGH INFO": "neutral",
    "entailment_score_alberta": "entailment_score"
}, axis="columns")

In [ ]:
def get_contradiction(dataframe, threshold=50):
    """Check if any of the reference contradict the answer"""
    contradiction = np.where(dataframe.contradiction >= threshold, True, False)
    if contradiction.any():
        return "CONTRADICTED"
    return "VALID"

In [ ]:
sentence_level_contradiction_labels = qa_pairs_df_xberta.groupby("Question").apply(
    get_contradiction, threshold=50)

In [ ]:
sentence_level_contradiction_labels.shape

In [ ]:
sentence_level_contradiction_labels.value_counts()

With xberat model, we have only 15 contradiction detected with our veto approach.

In [ ]:
sentence_level_contradiction_labels.head()

In [ ]:
sentence_level_contradiction = sentence_level_contradiction_labels.loc[
    sentence_level_contradiction_labels == "CONTRADICTED"].index.unique()

In [ ]:
sentence_level_contradiction.shape

In [ ]:
query_df

In [ ]:
import random

In [ ]:
test_question = "I have a client who has a buy to let mortgage which is rented out, will Halifax use the profits from this for resi affordability purposes?"

In [ ]:
sample_question = random.choice(sentence_level_contradiction)
query_df = qa_pairs_df_xberta[qa_pairs_df_xberta.Question == test_question]
test_pivot = query_df[["content", "generated_answer_sentence", "entailment_score"]].pivot(
    columns="generated_answer_sentence", index="content", values="entailment_score")
styled_df = test_pivot.style.map(color_cell, labels_to_color={
                                 1: '#f0a190', 0: '#cff090', 2: '#90d3f0'}).format(display_value)
print(f"Sample Question: {test_question}\n")
styled_df

In [ ]:
test_pivot

This model seems to be more coherent with the NLI lablels.

### Approach on the Answer directly without breaking it into chunks.

In [ ]:
answer_level_score = xlarge_model.predict(
    questions_df[["content", "generated_answer"]].values.tolist(),
    apply_softmax=True,
    batch_size=8)

In [ ]:
entailment_score = pd.DataFrame(answer_level_score.round(
    2) * 100, columns=xlarge_model.config.id2label.values())

In [ ]:
entailment_score["entailment_score"] = (
    answer_level_score.round(2) * 100).tolist()

In [ ]:
entailment_score.head()

In [ ]:
answer_level_df = pd.concat([questions_df[["Question", "generated_answer", "content"]].reset_index(
    drop=True), entailment_score], axis="columns")

In [ ]:
answer_level_df = answer_level_df.rename({
    "SUPPORTS": "entailment",
    "REFUTES": "contradiction",
    "NOT ENOUGH INFO": "neutral",
    "entailment_score_alberta": "entailment_score"
}, axis="columns")

In [ ]:
answer_level_contradiction_labels = answer_level_df.groupby(
    "Question").apply(get_contradiction, threshold=30)

In [ ]:
answer_level_contradiction = answer_level_contradiction_labels.loc[answer_level_contradiction_labels == "CONTRADICTED"]

In [ ]:
answer_level_contradiction

In [ ]:
answer_level_df.head()

In [ ]:
query_df

In [ ]:
sample_question = random.choice(answer_level_contradiction.index.unique())
query_df = answer_level_df[answer_level_df.Question == sample_question]
test_pivot = query_df[["content", "generated_answer", "entailment_score"]].pivot(
    columns="generated_answer", index="content", values="entailment_score")
styled_df = test_pivot.style.map(color_cell, labels_to_color={
                                 1: '#f0a190', 0: '#cff090', 2: '#90d3f0'}).format(display_value)
print(sample_question)
styled_df

In [ ]:
styled_df.to_html()

We can check the logic, all the reference contradict the answer, we can also switch and check if one reference contradict the answer

In [ ]:
answer_level_contradiction.index.unique()

In [ ]:
np.setdiff1d(sentence_level_contradiction.unique(),
             answer_level_contradiction.index.unique()).shape

In [ ]:
sentence_level_contradiction.shape

In [ ]:
answer_level_contradiction.index.unique().shape

In [ ]:
np.intersect1d(answer_level_contradiction.index.unique(),
             sentence_level_contradiction.unique())

When we put the thershold at 50% we only have 5 contradiction, compared to 10 from the previous approach, this number can be increase depending on how we set up the thershold

Let see those who are in when we split by sentence and those who are not in when 

Answer comparison between what was flag as hallucination and the correct answers.

In [ ]:
questions_df.head()

In [ ]:
answer_level_contradiction.shape

In [ ]:

with pd.option_context('display.max_rows', 0, 'display.max_columns', 0, 'display.width', 0, 'display.max_colwidth', 0):
    display(questions_df.loc[questions_df.Question.isin(answer_level_contradiction.index)][[
            "Question", "generated_answer", "Answer"]].drop_duplicates())

In [ ]:
qa_pairs_df["sentence_level_scores"] = (sentence_level_scores.round(2) * 100).tolist()


In [ ]:
questions_df["answer_level_scores"] = (answer_level_score.round(2) * 100).tolist()

In [ ]:
questions_df.head()

In [ ]:
questions_df[["Question", "content", "generated_answer", "Answer", "answer_level_scores"]].to_csv(
    data_directory.joinpath("nli_answer_level_scores.csv"), sep="|")

In [ ]:
qa_pairs_df[["Question", "content", "sentence_level_scores", "generated_answer_sentence"]].to_csv(
    data_directory.joinpath("nli_sentence_level_scores.csv"), sep="|")

In [ ]:
qa_pairs_df.head()

#### Vectara Stuff

In [ ]:
from transformers import AutoModelForSequenceClassification

In [ ]:
vectara_model = AutoModelForSequenceClassification.from_pretrained(
    'vectara/hallucination_evaluation_model', 
    trust_remote_code=True, 
    cache_dir=model_cache)

In [ ]:
questions_df["claim"] = "The Answer to the question: '" + questions_df["Question"] + "' is: '" + questions_df["generated_answer"] 

In [ ]:
vectara_model_score = vectara_model.predict(questions_df[["content", "claim"]].values)

In [ ]:
vectara_model_score.shape

In [ ]:
questions_df["vectara_model_scores"] = vectara_model_score

In [ ]:
vectara_thershold = 0.1
vectara_labels = questions_df.groupby("Question").vectara_model_scores.apply(
    lambda x: any(x > vectara_thershold))

In [ ]:
failed_answer_vectara = vectara_labels.loc[vectara_labels == False]

In [ ]:
sample_question = random.choice(failed_answer_vectara.index.unique())
query_df = questions_df[questions_df.Question == sample_question]
test_pivot = query_df[["content", "generated_answer", "vectara_model_scores"]].pivot(
    columns="generated_answer", index="content", values="vectara_model_scores")
print(sample_question)
test_pivot